<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_20236_1/blob/main/5_An%C3%A1lis_de_una_variable_Python_Notebook_y_RMarkdown_con_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este es un plan para reorganizar tu notebook en un formato más fluido, totalmente enfocado en **Python** (eliminando la dependencia de R Magic para las demostraciones centrales) y optimizado para una metodología de **aprendizaje asistido por IA**.

---

## Estructura Propuesta del Notebook (Python)

### 1. Introducción y Configuración

**Prompt para el estudiante:**

> "Actúa como un experto en Ciencia de Datos. Explica brevemente qué es el Análisis Exploratorio de Datos (EDA) unidimensional y por qué es fundamental identificar la distribución de una variable antes de realizar inferencia estadística. Resume los conceptos de distribuciones paramétricas y no paramétricas (KDE)."

In [ ]:
!pip install fitter

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from fitter import Fitter

# Configuración estética
sns.set_theme(style="whitegrid")

---

### 2. Demostración de la Probabilidad Frecuentista

**Prompt para el estudiante:**

> "Explica el concepto de Probabilidad Frecuentista o 'A Posteriori'. Define la fórmula del límite cuando $N$ tiende a infinito y explica cómo una simulación de Monte Carlo puede demostrar que la frecuencia relativa converge a la probabilidad teórica."

In [ ]:
# Simulación de control de aire acondicionado (6 unidades)
unidades = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']
n_simulaciones = [10, 100, 1000, 10000]

print("Convergencia de la Frecuencia Relativa (Probabilidad Teórica = 0.1666)\n")
for n in n_simulaciones:
    muestra = pd.Series(np.random.choice(unidades, size=n))
    frec_relativa = muestra.value_counts(normalize=True).sort_index()
    print(f"Resultados con N={n}:")
    print(frec_relativa.to_string(), "\n")

---

### 3. Tabla de Distribución de Frecuencias (TDF) en Python

**Prompt para el estudiante:**

> "Investiga la Regla de Sturges para calcular el número de intervalos (bins) en un histograma. ¿Cuáles son las diferencias entre frecuencia absoluta, relativa y acumulada? Pide a Gemini que explique la importancia de la 'Marca de Clase' en el análisis de datos agrupados."

In [ ]:
def generar_tdf(datos, nombre_var="Variable"):
    n = len(datos)
    k = int(1 + 3.322 * np.log10(n)) # Regla de Sturges
    rango = datos.max() - datos.min()
    ancho = np.ceil(rango / k)

    bins = [datos.min() + i*ancho for i in range(k+1)]

    tdf = pd.DataFrame()
    counts, edges = np.histogram(datos, bins=bins)

    tdf['Linf'] = edges[:-1]
    tdf['Lsup'] = edges[1:]
    tdf['mc'] = (tdf['Linf'] + tdf['Lsup']) / 2
    tdf['f'] = counts
    tdf['h'] = counts / n
    tdf['F'] = tdf['f'].cumsum()
    tdf['H'] = tdf['h'].cumsum()

    return tdf

# Ejemplo con datos de Boston (California Housing en Colab)
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)

print("Tabla de Frecuencias para 'MedInc' (Ingreso Medio):")
display(generar_tdf(df_housing['MedInc']))

---

### 4. Visualización: Histograma y Densidad (KDE)

**Prompt para el estudiante:**

> "¿Qué es la Estimación de Densidad de Kernel (KDE)? Explica cómo se construye la curva sumando gaussianas individuales y qué función cumple el parámetro 'bandwidth' (ancho de banda) en el suavizado de la curva."

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df_housing['MedInc'], kde=True, stat="density", color="skyblue")
plt.title("Distribución de Ingresos con Curva KDE")
plt.show()

---

### 5. Ajuste de Distribuciones Teóricas

**Prompt para el estudiante:**

> "Explica el proceso de 'ajuste de bondad' (Goodness of Fit). ¿Qué busca el test de Kolmogorov-Smirnov y cómo se interpreta el p-valor para decidir si un conjunto de datos sigue una distribución Normal o Gamma?"

In [ ]:
# Ajuste visual con Seaborn
data = df_housing['MedInc']
plt.figure(figsize=(10,6))
sns.distplot(data, kde=False, fit=stats.gamma, label="Ajuste Gamma")
sns.distplot(data, kde=False, fit=stats.norm, label="Ajuste Normal", hist_kws={"alpha":0.2})
plt.legend()
plt.show()

# Test de Kolmogorov-Smirnov
params = stats.norm.fit(data)
d, p_valor = stats.kstest(data, "norm", args=params)
print(f"P-valor (Normal): {p_valor}")
print("Resultado: Es Normal" if p_valor > 0.05 else "Resultado: No es Normal")

---

### 6. Selección Automática con Fitter

**Prompt para el estudiante:**

> "Documenta el uso de la librería 'Fitter'. ¿Cómo ayuda a automatizar la búsqueda de la mejor distribución entre las 80 disponibles en Scipy? Explica qué es el error cuadrático medio en este contexto."

In [ ]:
# Este paso puede tardar unos segundos
f = Fitter(data, distributions=['gamma', 'lognorm', 'norm', 'expon'])
f.fit()
f.summary()
print("\nMejor distribución:", f.get_best(method='sumsquare_error'))

---

## Guía para el Estudiante: De Python a RMarkdown

Para finalizar el ejercicio y cumplir con los requisitos de la asignatura, sigue estos pasos en tu entorno local o Posit Cloud:

### 1. Crear el archivo RMarkdown

Abre RStudio y crea un nuevo archivo `.Rmd`. Asegúrate de tener instalada la librería `reticulate` para ejecutar el código de Python que desarrollaste.

### 2. Estructura del Documento

```markdown
---
title: "Análisis de Distribuciones: Python + R"
author: "Tu Nombre"
output: html_document
---

# Introducción
En este documento se integra el análisis realizado en Python...

```{r setup}
library(reticulate)
# Esto permite que R use el entorno de Python

```

# Ejecución de Código Python

```{python}
# Aquí pegas el código de ajuste de distribuciones desarrollado arriba
import numpy as np
# ... (tu código)

```

# Publicación

1. **GitHub:** Sube tu archivo `.Rmd` y el `.html` resultante a un repositorio.
2. **RPubs:** En RStudio, haz clic en el botón **Publish** (esquina superior derecha de la consola) y selecciona RPubs.

```

### 3. Entregables
* **URL de GitHub:** Donde se aloja el código fuente.
* **URL de RPubs:** Donde se visualiza el reporte profesional renderizado.

```